# Amery appendix emulator diagnostics (A1–A3) + Fig 8 SLR projection

All predicted values here **exclude emulator / "code" uncertainty**:

* **A1 / A2** — 5-fold cross-validation fit of the per-year *cumulative grounded-mass-change* GP
  emulator at **2150 / 2300**: (a) residuals-vs-fitted, (b) predicted-vs-actual, (c) residual-coloured
  parameter-pair scatters. Panel (b) shows **no predictive-interval error bars** (that band was the
  emulator/code uncertainty; it is intentionally removed).
* **A3** — out-of-fold CV predictions (mean only, combined across folds) vs the simulated "actual"
  grounded-mass change, connected as lines across the 19 analysis years {2030, 2045, …, 2300} (Gt).
* **Fig 8** — posterior-predictive **sea-level contribution** at 2100/2200/2300 from pushing the
  present-day (2015) posterior through the VAF PCA-GP emulator, computed **both** with code
  uncertainty *off* (GP mean per PC) and *on* (the `rand(Normal(mean, σ))` PC draw), so the two can
  be compared directly.

## 1&nbsp;&middot;&nbsp;Imports &amp; plotting style

In [ ]:
using CSV, DataFrames
using GaussianProcesses, Optim
using StatsBase, Statistics, Distributions
using MultivariateStats
import MultivariateStats: reconstruct
using JLD2, LinearAlgebra, Random, Printf
using PyCall, PyPlot
BLAS.set_num_threads(1)   # bit-reproducibility must not depend on the caller's core count

const seaborn = pyimport("seaborn")
const np      = pyimport("numpy")
try
    pyimport("scienceplots")
    PyPlot.matplotlib.style.use(["default", "science", "no-latex"])
catch err
    @warn "scienceplots style unavailable; falling back to default matplotlib style." err
    PyPlot.matplotlib.style.use("default")
end

## 2&nbsp;&middot;&nbsp;Configuration &amp; paths

In [ ]:
# Locate the repo root robustly (works whether the kernel's pwd is notebooks/ or the repo root).
function find_root()
    d = pwd()
    for _ in 1:8
        (isdir(joinpath(d, "inputs")) && isfile(joinpath(d, "Project.toml"))) && return d
        d = dirname(d)
    end
    error("Could not locate repo root (an ancestor with inputs/ and Project.toml).")
end
const ROOT = find_root()

const PARAMS   = [:vmThresh, :fricExp, :mu_scale, :stiff_scale, :gamma0, :melt_flux]
const GAP      = 15
# analysis years 2030..2300 (override with FL_FIG_YEARS="2150,2300" for a quick smoke test)
FL_FIG_YEARS="2150,2300"
const ALL_YEARS = let e = get(ENV, "FL_FIG_YEARS", "")
    isempty(e) ? [y for y in 2015:GAP:2300 if y != 2015] : parse.(Int, strip.(split(e, ",")))
end

const N_FOLDS      = 5
const KF_SEED      = 42
const BASE_ITERS   = 100
const N_RETRIES    = 3
const LOG_OBS_NOISE = -4.605170185985     # log(0.01); held constant during fitting

const INPUT_CSV  = joinpath(ROOT, "inputs", "data", "Amery_Input_Parameters_Filtered.csv")
const OUTPUT_CSV = joinpath(ROOT, "inputs", "data", "Amery_Grd_Vol_Change_time_series_2300_ssp5.csv")
const EMU_JLD2   = joinpath(ROOT, "inputs", "emulators", "VAF_PCA_GP_Emulator.jld2")
const POST_NPY   = joinpath(ROOT, "inputs", "params", "present_day_2015_posterior_samples.npy")
const N_KEEP     = parse(Int, get(ENV, "FL_FIG_NKEEP", "5000"))       # posterior draws for Fig 8
const GT2MM      = -362                                               # VAF(Gt) -> SLR(mm)
const OUT_DIR    = get(ENV, "FL_FIG_OUTDIR", joinpath(ROOT, "outputs", "plots", "appendix_and_slr"))
mkpath(OUT_DIR)

# plot cosmetics (match the reference notebooks)
const SIZE      = 18
const FONTSIZE  = 22.5
const DOTCOLOR  = "green"
const EDGECOLOR = "gray"
println("ROOT     = ", ROOT)
println("OUT_DIR  = ", OUT_DIR)
println("years    = ", ALL_YEARS)

## 3&nbsp;&middot;&nbsp;Load &amp; scale the grounded-mass-change data

In [ ]:
include(joinpath(ROOT, "src", "utils", "scale_utils.jl"))
using .ScaleUtils

X_raw = CSV.read(INPUT_CSV, DataFrame)
for c in PARAMS
    @assert String(c) in names(X_raw) "Input CSV missing expected column :$c"
end
y_out_raw = CSV.read(OUTPUT_CSV, DataFrame)
Y_out = collect(Matrix(select(y_out_raw, Not(1))))          # N × 284 (drop index col)
@assert nrow(X_raw) == size(Y_out, 1) "Row mismatch: inputs $(nrow(X_raw)) vs outputs $(size(Y_out,1))."
const N_MEMBERS = nrow(X_raw)

# inputs scaled to [0,1] with the SAME transform the calibration uses; keep the min/max for Fig 8
X_scaled_NxD, _, X_mins, X_maxs = ScaleUtils.get_scaled_matrix(X_raw, PARAMS)
const X_SCALED = collect(Matrix(X_scaled_NxD)')             # 6 × N
const X_RAW    = collect(Matrix(select(X_raw, PARAMS))')         # 6 × N raw, for leak-free per-fold scaling

# raw parameter columns (original order) for the scatter panels
const P = (vmThresh   = Float64.(X_raw.vmThresh),   fricExp  = Float64.(X_raw.fricExp),
           mu_scale   = Float64.(X_raw.mu_scale),   stiff_scale = Float64.(X_raw.stiff_scale),
           gamma0     = Float64.(X_raw.gamma0),     melt_flux = Float64.(X_raw.melt_flux))
println("N_MEMBERS = ", N_MEMBERS, ",  output years = ", size(Y_out, 2))

## 4&nbsp;&middot;&nbsp;Cross-validation core (code uncertainty OFF)

Identical recipe to `src/diagnostics/emulator_diagnostics.jl` (sklearn-matched `KFold`, linear-mean
Matérn-5/2 ARD GP, fixed 0.01 nugget during fitting). `cv_year` keeps only the **out-of-fold
posterior mean** (`predict_f` mean) — the predictive variance is never used, so nothing "code" enters
the predicted values.

In [ ]:
# sklearn KFold(n_splits, shuffle=true, random_state=seed), reproduced byte-for-byte -> 1-based idx
function kfold_splits(n::Int, k::Int, seed::Int)
    rng = np.random.RandomState(seed)
    idx = np.arange(n); rng.shuffle(idx)
    perm = Int.(idx) .+ 1
    base, rem = divrem(n, k)
    sizes = fill(base, k); sizes[1:rem] .+= 1
    splits = Tuple{Vector{Int},Vector{Int}}[]; cur = 1
    for fs in sizes
        te = perm[cur:cur+fs-1]; tr = vcat(perm[1:cur-1], perm[cur+fs:end])
        push!(splits, (tr, te)); cur += fs
    end
    return splits
end

# one fold's GP: linear mean + Matern-5/2 ARD, fixed noise; double iters up to N_RETRIES on non-conv.
function fit_fold_gp(Xtr::AbstractMatrix, ytr::AbstractVector)
    d = size(Xtr, 1); iters = BASE_ITERS; local gp
    for attempt in 0:N_RETRIES
        gp = GaussianProcesses.GP(Xtr, ytr, MeanLin(zeros(d)), Mat52Ard(zeros(d), 0.001), LOG_OBS_NOISE)
        res = optimize!(gp; domean=true, kern=true, noise=false, iterations=iters)
        Optim.converged(res) && return gp
        attempt < N_RETRIES ? (iters *= 2) :
            @warn "fold GP did not converge after $(N_RETRIES+1) tries (up to $iters iters); using last state."
    end
    return gp
end

# out-of-fold CV for one year -> (actual, fitted mean, residual) in Gt (mean only: code unc OFF).
# Both scalers (input min–max per feature and output min–max) are fit on the TRAIN fold ONLY inside
# each split, so held-out members never influence preprocessing (no CV leakage).
function cv_year(year::Int)
    ycol = year - 2016
    @assert 1 <= ycol <= size(Y_out, 2) "Year $year -> column $ycol out of range."
    y_actual = Float64.(Y_out[:, ycol])
    pred = zeros(N_MEMBERS)
    for (tr, te) in kfold_splits(N_MEMBERS, N_FOLDS, KF_SEED)
        xmin  = vec(minimum(X_RAW[:, tr]; dims = 2)); xmax = vec(maximum(X_RAW[:, tr]; dims = 2))
        xspan = ifelse.(xmax .== xmin, 1.0, xmax .- xmin)
        Xtr   = (X_RAW[:, tr] .- xmin) ./ xspan
        Xte   = (X_RAW[:, te] .- xmin) ./ xspan
        ymin, ymax = extrema(@view y_actual[tr]); yspan = ymax == ymin ? 1.0 : ymax - ymin
        ytr   = (y_actual[tr] .- ymin) ./ yspan
        gp   = fit_fold_gp(Xtr, ytr)
        μ, _ = GaussianProcesses.predict_f(gp, Xte)               # mean only (code unc OFF)
        pred[te] = μ .* yspan .+ ymin                             # unscale with the train output scaler
    end
    residual = y_actual .- pred
    return (actual = y_actual, fitted = pred, residual = residual)
end

## 5&nbsp;&middot;&nbsp;Run 5-fold CV for every analysis year

This is the compute-heavy step (5 GP refits × each year). The per-year RMSE / R² printed below
match `emulator_diagnostics.jl` (e.g. 2300 → RMSE ≈ 2035 Gt, R² ≈ 0.976), since the predicted means
are identical.

In [ ]:
Random.seed!(42)
cv = Dict{Int,NamedTuple}()
@printf("%-6s %14s %10s\n", "year", "RMSE (Gt)", "R2")
for y in sort(ALL_YEARS)
    r = cv_year(y); cv[y] = r
    rmse   = sqrt(mean(r.residual .^ 2))
    ss_res = sum(r.residual .^ 2); ss_tot = sum((r.actual .- mean(r.actual)) .^ 2)
    @printf("%-6d %14.3f %10.5f\n", y, rmse, ss_tot == 0 ? NaN : 1 - ss_res / ss_tot)
end

## 6&nbsp;&middot;&nbsp;A1 / A2 — grounded-mass CV 6-panel (no predictive intervals)

Same layout/labels as the paper's Fig A1/A2, but panel (b) is a plain predicted-vs-actual scatter
with the 45° line — **the error bars are removed** because they encoded the emulator/code uncertainty.
No figure title.

In [ ]:
function plot_cv_6panel(year; fname = nothing, clip = nothing)
    haskey(cv, year) || error("year $year not in cv; add it to ALL_YEARS / FL_FIG_YEARS and re-run the CV cell.")
    r = cv[year]; actual, fitted, residual = r.actual, r.fitted, r.residual
    cmax = maximum(abs, residual); cmax = cmax == 0 ? 1.0 : cmax
    # COLOUR-scale cap (panel-(a) axis stays full-range): default 95th percentile of |residual| so
    # outliers saturate to the colormap extremes and smaller residuals get more contrast (Fig A1/A2).
    ccap = clip === nothing ? quantile(abs.(residual), 0.95) : Float64(clip)
    ccap = ccap == 0 ? cmax : ccap
    @printf("  %d: colour cap = %.4g Gt  (max|resid| = %.4g Gt)\n", year, ccap, cmax)
    lo, hi = extrema(vcat(actual, fitted)); pad = 0.05 * (hi - lo == 0 ? 1.0 : hi - lo); lo -= pad; hi += pad
    flo, fhi = extrema(fitted); fpad = 0.05 * (fhi - flo == 0 ? 1.0 : fhi - flo); flo -= fpad; fhi += fpad

    fig, ax = subplots(nrows = 2, ncols = 3, figsize = (16, 10))

    # (a) residuals vs fitted with lowess smooth
    try
        seaborn.regplot(x = fitted, y = residual, lowess = true,
            scatter_kws = Dict("alpha"=>0.85, "s"=>SIZE*2, "color"=>DOTCOLOR, "edgecolor"=>EDGECOLOR,
                               "linewidths"=>0.5, "plotnonfinite"=>false, "zorder"=>2),
            line_kws = Dict("color"=>"red", "lw"=>1, "alpha"=>0.8), ax = ax[1,1])
    catch err
        @warn "seaborn lowess failed; plain scatter fallback." year err
        ax[1,1].scatter(fitted, residual, s=SIZE*2, c=DOTCOLOR, alpha=0.85, edgecolor=EDGECOLOR, linewidths=0.5, zorder=2)
    end
    ax[1,1].set_ylabel("Residuals", fontsize=FONTSIZE)
    ax[1,1].set_xlabel("Fitted values", fontsize=FONTSIZE, labelpad=10)
    ax[1,1].grid(true, alpha=0.5, zorder=1)
    ax[1,1].ticklabel_format(style="sci", scilimits=(-4,4), useMathText=true)
    ax[1,1].tick_params(axis="both", which="major", labelsize=FONTSIZE*0.8)
    ax[1,1].xaxis.offsetText.set_fontsize(FONTSIZE*0.8); ax[1,1].yaxis.offsetText.set_fontsize(FONTSIZE*0.8)
    ax[1,1].set_ylim(-1.1*cmax, 1.1*cmax); ax[1,1].set_xlim(flo, fhi); ax[1,1].locator_params(tight=true, nbins=6)
    # ax[1,1].axhline(0, color="black", lw=0.8, alpha=0.7, zorder=1)
    ax[1,1].text(-0.2, 1, "(a)", ha="left", va="top", transform=ax[1,1].transAxes, fontsize=FONTSIZE)

    # (b) predicted vs actual -- NO error bars (code uncertainty off)
    ax[2,1].scatter(x=actual, y=fitted, s=SIZE*2, c=DOTCOLOR, alpha=0.85,
                    edgecolor=EDGECOLOR, linewidths=0.5, plotnonfinite=false, zorder=4)
    ax[2,1].plot([hi, lo], [hi, lo], "r-")
    ax[2,1].set_xlabel("True Values", fontsize=FONTSIZE)
    ax[2,1].set_ylabel("Fitted values", fontsize=FONTSIZE, labelpad=10)
    ax[2,1].set_xlim(lo, hi); ax[2,1].set_ylim(lo, hi); ax[2,1].grid(true, alpha=0.5, zorder=1)
    ax[2,1].ticklabel_format(style="sci", scilimits=(-3,3), useMathText=true)
    ax[2,1].tick_params(axis="both", which="major", labelsize=FONTSIZE*0.8)
    ax[2,1].xaxis.offsetText.set_fontsize(FONTSIZE*0.8); ax[2,1].yaxis.offsetText.set_fontsize(FONTSIZE*0.8)
    ax[2,1].locator_params(tight=true, nbins=6)
    ax[2,1].text(-0.2, 1, "(b)", ha="left", va="top", transform=ax[2,1].transAxes, fontsize=FONTSIZE)

    # (c)-(f) residual-coloured parameter-pair scatters (shared symmetric colour range)
    function pj(a, xv, yv, xl, yl; tag = nothing)
        im = a.scatter(xv, yv, s=SIZE*5, c=residual, edgecolor=EDGECOLOR, linewidths=0.5,
                       plotnonfinite=false, cmap="RdBu", vmin=-ccap, vmax=ccap, zorder=2)
        a.set_xlabel(xl, fontsize=FONTSIZE, labelpad=10); a.set_ylabel(yl, fontsize=FONTSIZE, labelpad=10)
        a.grid(true, alpha=0.5, zorder=1); a.locator_params(tight=true, nbins=6)
        a.ticklabel_format(style="sci", scilimits=(-3,3), useMathText=true)
        a.tick_params(axis="both", which="major", labelsize=FONTSIZE*0.8)
        a.xaxis.offsetText.set_fontsize(FONTSIZE*0.8); a.yaxis.offsetText.set_fontsize(FONTSIZE*0.8)
        tag !== nothing && a.text(-0.24, 1, tag, ha="left", va="top", transform=a.transAxes, fontsize=FONTSIZE)
        return im
    end
    im = pj(ax[1,2], P.mu_scale,    P.fricExp,  L"C_{\mu}",     L"q"; tag="(c)")
         pj(ax[1,3], P.stiff_scale, P.vmThresh, L"C_{\phi}",    L"\sigma_{max}")
         pj(ax[2,2], P.stiff_scale, P.mu_scale, L"C_{\phi}",    L"C_{\mu}")
         pj(ax[2,3], P.melt_flux,   P.gamma0,   L"\overline{m}", L"\gamma_0")

    plt.tight_layout()
    cax  = fig.add_axes([1.03, 0.25, 0.02, 0.5])
    cbar = fig.colorbar(im, cax=cax); cbar.mappable.set_clim(-ccap, ccap)
    cbar.ax.yaxis.offsetText.set_fontsize(FONTSIZE*0.8)
    cbar.ax.yaxis.offsetText.set_position((1.3, 0))
    cbar.ax.ticklabel_format(style="sci", scilimits=(-3,3), useMathText=true)
    cbar.ax.locator_params(tight=true, nbins=6)
    cbar.ax.set_title("Residuals", fontsize=FONTSIZE, pad=32)
    cbar.ax.tick_params(axis="both", which="major", labelsize=FONTSIZE*0.8)

    fname !== nothing && fig.savefig(joinpath(OUT_DIR, fname), bbox_inches="tight", dpi=300)
    return fig
end

In [ ]:
# plot_cv_6panel(2150; fname = "A1_grd_mass_cv_2150.pdf", clip=2000);   # Figure A1

In [ ]:
# plot_cv_6panel(2300; fname = "A2_grd_mass_cv_2300.pdf", clip=5000);   # Figure A2

## 7&nbsp;&middot;&nbsp;A3 — CV predictions vs actual across analysis years

For every realization, the out-of-fold predicted (mean) and actual cumulative grounded-mass change
are connected across {2030…2300}. Predicted = dashed blue, actual = solid salmon.

In [ ]:
Ys   = sort(ALL_YEARS)
FITm = hcat([cv[y].fitted for y in Ys]...)   # N × nYears
ACTm = hcat([cv[y].actual for y in Ys]...)   # N × nYears

FONTSIZE_A3 = 14
fig, ax = subplots(figsize = (8, 5))
for i in 1:N_MEMBERS
    ax.plot(Ys, ACTm[i, :], color="salmon",    lw=0.8, alpha=0.8, label = i == 1 ? "actual" : "")
    ax.plot(Ys, FITm[i, :], color="dodgerblue", lw=0.8, ls="-", alpha=0.8, label = i == 1 ? "predicted" : "")
end
ax.set_xlabel("Year", fontsize=FONTSIZE_A3, labelpad=10)
ax.set_ylabel("Cumulative grounded mass change (Gt)", fontsize=FONTSIZE_A3)
ax.tick_params(axis="both", which="major", labelsize=FONTSIZE_A3*0.8)
ax.set_xticks([2050, 2100, 2150, 2200, 2250, 2300]); ax.set_xlim(minimum(Ys), maximum(Ys))
ax.grid(true, alpha=0.5, zorder=1)
ax.ticklabel_format(axis="y", style="sci", scilimits=(-3,3), useMathText=true)
ax.yaxis.offsetText.set_fontsize(FONTSIZE_A3*0.8)
leg = ax.legend(fontsize=0.9*FONTSIZE_A3, frameon=true, framealpha=1, loc="lower left", bbox_to_anchor=(0.051, 0.072))
handles = try leg.legend_handles catch; leg.legendHandles end   # matplotlib >=3.9 renamed this
for h in handles; h.set_alpha(1); h.set_linewidth(2); end
# fig.savefig(joinpath(OUT_DIR, "A3_grd_mass_validation_across_years.pdf"), bbox_inches="tight", dpi=300)

## 8&nbsp;&middot;&nbsp;Fig 8 — SLR posterior push-forward

Push the present-day (2015) posterior through the VAF PCA-GP emulator using the GP **mean** per PC
(emulator/code uncertainty excluded), reconstruct, and convert VAF (Gt) → sea-level (mm).

In [ ]:
gps_vaf, PC_min, PC_max, pca, num_pcs = JLD2.load(EMU_JLD2, "gps", "PC_min", "PC_max", "p", "num_pcs")
predf(gp, X) = (mv = GaussianProcesses.predict_f(gp, X); (mv[1], sqrt.(mv[2])))
reconstruct_pcs(pca, W, J=length(W)) =
    MultivariateStats.decentralize(MultivariateStats.projection(pca)[:, 1:J] * W, mean(pca))

param_mins = [X_mins[p] for p in PARAMS]
param_maxs = [X_maxs[p] for p in PARAMS]
chain = convert(Matrix{Float64}, np.load(POST_NPY))            # 100000 × 6 (physical, PARAMS order)
step  = max(1, fld(size(chain, 1), N_KEEP))
draws = chain[1:step:end, :]; size(draws, 1) > N_KEEP && (draws = draws[1:N_KEEP, :])
αs    = (draws .- param_mins') ./ (param_maxs' .- param_mins')  # N × 6 scaled to [0,1]
ntime = size(MultivariateStats.projection(pca), 1)             # 285

push_off = Matrix{Float64}(undef, ntime, size(draws, 1))       # SLR per posterior draw (GP mean per PC)
for s in 1:size(αs, 1)
    α = reshape(αs[s, :], :, 1)
    pc_mean = Vector{Float64}(undef, num_pcs)
    for i in 1:num_pcs
        μ, _ = predf(gps_vaf[i], α)
        pc_mean[i] = only(μ) * (PC_max[i] - PC_min[i]) + PC_min[i]
    end
    push_off[:, s] = reconstruct_pcs(pca, pc_mean, num_pcs)
end
push_off ./= GT2MM                            # -> sea-level contribution (mm)
println("push-forward done: ", size(push_off), " (year × sample), mm")

## 9&nbsp;&middot;&nbsp;Fig 8 — posterior-predictive sea-level distributions

In [ ]:
# Faithful port of the paper's Fig 8 cell (ESD_Paper_Figure), with code uncertainty OFF (push_off):
# KDE per year + 90% credible-interval fill (5th-95th percentile) + inset zoom of the 2200/2300 tails.
start_year = 2015
FS8   = 18
years8 = [2100, 2200, 2300]; cols8 = ["green", "red", "purple"]; fills8 = ["palegreen", "coral", "plum"]
preds8 = [push_off[y - start_year, :] for y in years8]
los8   = [np.percentile(p, 5)  for p in preds8]
his8   = [np.percentile(p, 95) for p in preds8]
for (y, p, lo, hi) in zip(years8, preds8, los8, his8)
    @printf("%d:  mean = %7.2f   5%% = %7.2f   95%% = %7.2f   (90%% width %6.2f) mm\n", y, mean(p), lo, hi, hi - lo)
end

fig, ax = subplots(figsize = (11, 6))
for (y, p, lo, hi, c, fc) in zip(years8, preds8, los8, his8, cols8, fills8)
    seaborn.kdeplot(p, lw=1.5, label="$y", color=c, ax=ax)
    seaborn.kdeplot(p, fill=true, clip=(lo, hi), color=fc, ax=ax)
end
seaborn.kdeplot(preds8[1], fill=true, clip=(-50.0, -50.0), color="gray", label="90% Cred Int", ax=ax)  # legend proxy
ax.set_ylabel("Density", fontsize=FS8, labelpad=15)
ax.set_xlabel("Sea-level contribution (mm)", fontsize=FS8, labelpad=10)
ax.tick_params(axis="both", which="major", labelsize=FS8*0.8)
ax.locator_params(tight=true, nbins=5); ax.set_xticks([0, 50, 100, 150]); ax.set_xlim(-27, 180.5)
ax.grid(true, alpha=0.5, zorder=1)
legend1 = ax.legend(fontsize=0.85*FS8, frameon=true, ncol=2, framealpha=1, bbox_to_anchor=(0.464, 0.94))
ax.add_artist(legend1)

# inset zoom of the 2200 & 2300 tails (as in the paper figure)
axins = ax.inset_axes([0.48, 0.254, 0.4, 0.4])
axins.set_xlim(-26, 179.4)   # y-limit is auto-fitted below, after the curves are drawn
for (p, lo, hi, c, fc) in zip(preds8[2:3], los8[2:3], his8[2:3], cols8[2:3], fills8[2:3])
    seaborn.kdeplot(p, lw=1.5, color=c, ax=axins)
    seaborn.kdeplot(p, fill=true, clip=(lo, hi), color=fc, ax=axins)
end
# auto-fit the inset y-limit to the tallest curve actually drawn (seaborn builds the KDE internally,
# so the peak is read back off the plotted lines). Replaces the paper cell's hardcoded 0.0295, which
# clipped the 2200 peak (0.02969) in the code-off variant.
_pk = maximum(maximum(l.get_ydata()) for l in axins.get_lines())
axins.set_ylim(0, 1.10 * _pk)
axins.locator_params(tight=true, nbins=5); axins.set_xticks([0, 50, 100, 150])
axins.tick_params(axis="both", which="major", labelsize=FS8*0.7)
axins.grid(true, alpha=0.5, zorder=1); axins.set_ylabel("")
ax.indicate_inset_zoom(axins, edgecolor=EDGECOLOR, fill=true, lw=1)

fig.savefig(joinpath(OUT_DIR, "Figure8_slr_posterior_kde_codeoff.pdf"), bbox_inches="tight", dpi=300)

### Outputs (written to `OUT_DIR`)
`A1_grd_mass_cv_2150.pdf`, `A2_grd_mass_cv_2300.pdf`, `A3_grd_mass_validation_across_years.pdf`,
`Figure8_slr_posterior_kde_codeoff.pdf`.